# Cookbook Experiment 1: Simple CNN Baseline (No Residual, No Skip)

## Why this notebook matters

This experiment is your baseline proof of the vanishing-gradient issue in a deep plain convolutional network. There is no residual addition, so gradients must pass only through chained nonlinear layers during backpropagation.

## Core concept tested

For a deep network with loss $\mathcal{L}$ and parameters $W_l$ at layer $l$:

$$\frac{\partial \mathcal{L}}{\partial W_l} = \frac{\partial \mathcal{L}}{\partial a_L}\prod_{i=l}^{L-1}\frac{\partial a_{i+1}}{\partial a_i}$$

When many factors in the product are small, early-layer gradients and updates shrink. This notebook records that effect per convolution layer.

## What you will produce from this notebook

- Training/test curves
- Per-layer gradient norms (batch and epoch)
- Per-layer weight norms and weight deltas
- Gradient heatmap and gradient ratio
- Confusion matrix and sample predictions
- Table-style summaries (architecture, performance, epoch metrics, gradient summary)
- ONNX model and architecture SVG

## Expected interpretation

- Early layers in plain CNN should tend to weaker updates than deeper layers.
- This creates a reference point before introducing residual structure and skip addition in later notebooks.

In [ ]:
import copy
import json
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from IPython.display import Image, display

SVHN_MEAN = [0.4377, 0.4438, 0.4728]
SVHN_STD = [0.1980, 0.2010, 0.1970]

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_svhn_loaders(data_dir="../dataset/SVHN", batch_size=128, num_workers=0):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=SVHN_MEAN, std=SVHN_STD),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=SVHN_MEAN, std=SVHN_STD),
    ])
    train_ds = datasets.SVHN(root=data_dir, split="train", download=True, transform=transform_train)
    test_ds = datasets.SVHN(root=data_dir, split="test", download=True, transform=transform_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

def unnormalize_image_tensor(x):
    mean = torch.tensor(SVHN_MEAN, dtype=x.dtype, device=x.device).view(3, 1, 1)
    std = torch.tensor(SVHN_STD, dtype=x.dtype, device=x.device).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)

class ConvBnRelu(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = ConvBnRelu(3, 64)
        layers = []
        channels = [64, 64, 64, 128, 128, 128, 256, 256, 256]
        in_ch = 64
        for idx, out_ch in enumerate(channels):
            stride = 2 if idx in (3, 6) else 1
            layers.append(ConvBnRelu(in_ch, out_ch, stride=stride))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

def get_model(config_name, num_classes=10):
    if config_name == "simple_cnn":
        return SimpleCNN(num_classes=num_classes)
    raise ValueError(f"Unknown config: {config_name}")

def get_tracked_conv_layers(model):
    tracked = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            tracked[name] = module
    return tracked

def _train_one_epoch(model, loader, criterion, optimizer, tracked_layers, history, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    batch_grad_acc = defaultdict(list)

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        for name, layer in tracked_layers.items():
            if layer.weight.grad is not None:
                g = layer.weight.grad.data.norm(2).item()
                batch_grad_acc[name].append(g)
                history["batch_grad_norms"][name].append(g)

        optimizer.step()
        running_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()

    for name in tracked_layers:
        vals = batch_grad_acc[name]
        history["epoch_grad_norms"][name].append(float(np.mean(vals)) if vals else 0.0)

    return running_loss / len(loader), 100.0 * correct / total

def _evaluate(model, loader, criterion, device):
    model.eval()
    loss_total, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_total += loss.item()
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
    return loss_total / len(loader), 100.0 * correct / total, y_true, y_pred

def _log_weight_stats(history, tracked_layers, prev_weights):
    for name, layer in tracked_layers.items():
        w = layer.weight.data
        history["weight_norms"][name].append(w.norm(2).item())
        history["weight_deltas"][name].append((w - prev_weights[name]).norm(2).item())
        prev_weights[name] = w.clone()

def export_to_onnx(model, save_path, device, input_size=(1, 3, 32, 32)):
    model.eval()
    dummy_input = torch.randn(*input_size, device=device)
    torch.onnx.export(
        model, dummy_input, save_path, export_params=True, opset_version=11,
        do_constant_folding=True, input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
    )

def _plot_multi_series(ax, data_dict, title, ylabel, logy=False):
    for name, values in data_dict.items():
        ax.plot(values, label=name, linewidth=1.1)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    if logy:
        ax.set_yscale("log")
    ax.grid(alpha=0.3)

def plot_training_curves(history, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["train_loss"], label="Train Loss")
    ax.plot(history["test_loss"], label="Test Loss")
    ax.set_title("Loss Curves")
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "loss_curves.png"), dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["train_acc"], label="Train Acc")
    ax.plot(history["test_acc"], label="Test Acc")
    ax.set_title("Accuracy Curves")
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "accuracy_curves.png"), dpi=150)
    plt.close(fig)

def plot_gradient_curves(history, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_multi_series(ax, history["epoch_grad_norms"], "Epoch Gradient Norms (All Conv Layers)", "L2 Norm", logy=True)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "epoch_gradient_norms.png"), dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_multi_series(ax, history["batch_grad_norms"], "Batch Gradient Norms (All Conv Layers)", "L2 Norm", logy=True)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "batch_gradient_norms.png"), dpi=150)
    plt.close(fig)

def plot_weight_curves(history, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_multi_series(ax, history["weight_norms"], "Weight Norms", "L2 Norm")
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "weight_norms.png"), dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_multi_series(ax, history["weight_deltas"], "Weight Deltas", "L2 Delta", logy=True)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "weight_deltas.png"), dpi=150)
    plt.close(fig)

def plot_gradient_heatmap(history, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    layers = list(history["epoch_grad_norms"].keys())
    matrix = np.array([history["epoch_grad_norms"][k] for k in layers], dtype=float)
    matrix = np.log10(np.maximum(matrix, 1e-12))
    fig, ax = plt.subplots(figsize=(10, max(5, len(layers) * 0.25)))
    im = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title("Gradient Heatmap (log10)")
    ax.set_xlabel("Epoch")
    ax.set_yticks(np.arange(len(layers)))
    ax.set_yticklabels(layers, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "gradient_heatmap.png"), dpi=150)
    plt.close(fig)

def plot_gradient_ratio(history, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    keys = list(history["epoch_grad_norms"].keys())
    first_key, last_key = keys[0], keys[-1]
    first = np.array(history["epoch_grad_norms"][first_key], dtype=float)
    last = np.array(history["epoch_grad_norms"][last_key], dtype=float)
    ratio = first / np.maximum(last, 1e-12)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(ratio, marker="o")
    ax.set_title(f"Gradient Ratio: {first_key} / {last_key}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Ratio")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "gradient_ratio.png"), dpi=150)
    plt.close(fig)

def plot_confusion(y_true, y_pred, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(10)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title("Confusion Matrix")
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "confusion_matrix.png"), dpi=150)
    plt.close(fig)

def plot_sample_images(loader, output_dir):
    plots_dir = os.path.join(output_dir, "plots")
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    fig.suptitle("SVHN Training Samples", fontsize=13, fontweight="bold")
    for i, ax in enumerate(axes.flat):
        img = unnormalize_image_tensor(images[i]).permute(1, 2, 0).cpu().numpy()
        ax.imshow(img)
        ax.set_title(f"Label: {int(labels[i])}", fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "sample_images.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

def plot_sample_predictions(model, loader, output_dir, device):
    plots_dir = os.path.join(output_dir, "plots")
    model.eval()
    images, labels = next(iter(loader))
    images = images[:16].to(device)
    labels = labels[:16].to(device)
    with torch.no_grad():
        preds = model(images).argmax(dim=1)
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    fig.suptitle("Sample Predictions", fontsize=13, fontweight="bold")
    for i, ax in enumerate(axes.flat):
        img = unnormalize_image_tensor(images[i]).permute(1, 2, 0).cpu().numpy()
        ax.imshow(img)
        ok = int(preds[i]) == int(labels[i])
        ax.set_title(f"T:{int(labels[i])} P:{int(preds[i])}", fontsize=9, color=("green" if ok else "red"))
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "sample_predictions.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

def generate_summary_tables(history, output_dir, model_title):
    plots_dir = os.path.join(output_dir, "plots")
    layers = history["tracked_layers"]
    first_layer, last_layer = layers[0], layers[-1]

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.axis("off")
    cols = ["Layer Group", "Output Size", "Block Type", "Operation"]
    rows = [
        ["initial", "32x32", "Conv stem", "Conv + BN + ReLU"],
        ["stage1", "32x32", "3 blocks", "2x Conv(64) per block"],
        ["stage2", "16x16", "3 blocks", "2x Conv(128) per block"],
        ["stage3", "8x8", "3 blocks", "2x Conv(256) per block"],
        ["head", "1x1 -> 10", "Pooling + FC", "AdaptiveAvgPool + Linear"],
    ]
    table = ax.table(cellText=rows, colLabels=cols, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.0)
    for j in range(len(cols)):
        table[0, j].set_facecolor("#2c3e50")
        table[0, j].set_text_props(color="white", fontweight="bold")
    ax.set_title(f"Table 1. Architecture Summary - {model_title}", fontsize=13, fontweight="bold", pad=14)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "table1_architecture.png"), dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.axis("off")
    best_test = max(history["test_acc"])
    rows = [[model_title, f"{history['train_acc'][-1]:.2f}", f"{best_test:.2f}", f"{100.0 - best_test:.2f}", str(len(layers))]]
    cols = ["Model", "Final Train Acc (%)", "Best Test Acc (%)", "Test Error (%)", "Tracked Conv Layers"]
    table = ax.table(cellText=rows, colLabels=cols, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.0)
    for j in range(len(cols)):
        table[0, j].set_facecolor("#2c3e50")
        table[0, j].set_text_props(color="white", fontweight="bold")
        table[1, j].set_text_props(fontweight="bold")
    ax.set_title(f"Table 2. Performance Summary - {model_title}", fontsize=13, fontweight="bold", pad=14)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "table2_performance.png"), dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(16, 9))
    ax.axis("off")
    rows = []
    for i in range(len(history["train_loss"])):
        rows.append([
            str(i + 1), f"{history['train_loss'][i]:.4f}", f"{history['train_acc'][i]:.2f}",
            f"{history['test_loss'][i]:.4f}", f"{history['test_acc'][i]:.2f}",
            f"{history['epoch_grad_norms'][first_layer][i]:.6f}", f"{history['epoch_grad_norms'][last_layer][i]:.6f}",
            f"{history['weight_deltas'][first_layer][i]:.6f}", f"{history['weight_deltas'][last_layer][i]:.6f}",
        ])
    cols = [
        "Epoch", "Train Loss", "Train Acc", "Test Loss", "Test Acc",
        f"Grad ({first_layer})", f"Grad ({last_layer})",
        f"Delta ({first_layer})", f"Delta ({last_layer})",
    ]
    table = ax.table(cellText=rows, colLabels=cols, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.35)
    for j in range(len(cols)):
        table[0, j].set_facecolor("#2c3e50")
        table[0, j].set_text_props(color="white", fontweight="bold")
    ax.set_title(f"Table 3. Epoch Metrics - {model_title}", fontsize=13, fontweight="bold", pad=14)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "table3_training_metrics.png"), dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(15, max(5, 0.35 * len(layers) + 2)))
    ax.axis("off")
    key_epochs = [0, max(0, len(history['train_loss'])//4 - 1), max(0, len(history['train_loss'])//2 - 1), max(0, 3*len(history['train_loss'])//4 - 1), len(history['train_loss']) - 1]
    key_epochs = sorted(set(key_epochs))
    cols = ["Layer"] + [f"Epoch {k+1}" for k in key_epochs] + ["Trend"]
    rows = []
    for layer in layers:
        vals = history["epoch_grad_norms"][layer]
        trend = "decreasing" if vals[-1] < vals[0] else "increasing"
        rows.append([layer] + [f"{vals[k]:.6f}" for k in key_epochs] + [trend])
    table = ax.table(cellText=rows, colLabels=cols, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)
    for j in range(len(cols)):
        table[0, j].set_facecolor("#2c3e50")
        table[0, j].set_text_props(color="white", fontweight="bold")
    ax.set_title(f"Table 4. Gradient Summary - {model_title}", fontsize=13, fontweight="bold", pad=14)
    fig.tight_layout()
    fig.savefig(os.path.join(plots_dir, "table4_gradient_summary.png"), dpi=180, bbox_inches="tight")
    plt.close(fig)

def draw_architecture_svg(config_name, output_dir, model_title):
    models_dir = os.path.join(output_dir, "models")
    os.makedirs(models_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 14))
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 14)
    ax.axis("off")
    ax.set_aspect("equal")

    def draw_block(y, label, sublabel, color, height=0.7):
        rect = patches.FancyBboxPatch((1, y), 4, height, boxstyle="round,pad=0.08", linewidth=1.2, edgecolor="#444", facecolor=color)
        ax.add_patch(rect)
        ax.text(3, y + height/2 + 0.08, label, ha="center", va="center", fontsize=9.5, fontweight="bold")
        ax.text(3, y + height/2 - 0.18, sublabel, ha="center", va="center", fontsize=7.5, color="#555")

    def draw_arrow(y_from, y_to):
        ax.annotate("", xy=(3, y_to + 0.005), xytext=(3, y_from), arrowprops=dict(arrowstyle="->", color="#666", lw=1.2))

    blocks = [
        (12.2, "Conv + BN + ReLU", "3->64, 32x32", "#dfe6e9"),
        (10.9, "Stage 1", "3 blocks, 64 ch, no skip", "#d6eaf8"),
        (9.6, "Stage 2", "3 blocks, 128 ch, no skip", "#d5f5e3"),
        (8.3, "Stage 3", "3 blocks, 256 ch, no skip", "#fdebd0"),
        (7.0, "GlobalAvgPool", "1x1", "#dfe6e9"),
        (5.7, "FC", "256 -> 10", "#dfe6e9"),
    ]
    prev_top = 13.2
    ax.text(3, prev_top + 0.3, "Input 3x32x32", ha="center", va="center", fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="#ecf0f1", edgecolor="#aaa"))
    for y, label, sub, color in blocks:
        draw_arrow(prev_top, y + 0.7)
        draw_block(y, label, sub, color)
        prev_top = y
    ax.text(3, 4.4, "Plain Stack: no residual blocks", ha="center", fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="#f4f6f6", edgecolor="#95a5a6"))
    ax.set_title(f"{model_title} - Architecture", fontsize=10, fontweight="bold", pad=8)
    svg_path = os.path.join(models_dir, f"_{config_name}.onnx.svg")
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
    plt.close(fig)

def train_and_analyze(config_name, output_dir, model_title, epochs=20, batch_size=128, learning_rate=1e-3, data_dir="../dataset/SVHN"):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "plots"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "models"), exist_ok=True)

    device = get_device()
    train_loader, test_loader = get_svhn_loaders(data_dir=data_dir, batch_size=batch_size)
    model = get_model(config_name).to(device)
    tracked_layers = get_tracked_conv_layers(model)

    history = {
        "config_name": config_name,
        "epochs": epochs,
        "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [],
        "tracked_layers": list(tracked_layers.keys()),
        "epoch_grad_norms": {k: [] for k in tracked_layers},
        "batch_grad_norms": {k: [] for k in tracked_layers},
        "weight_norms": {k: [] for k in tracked_layers},
        "weight_deltas": {k: [] for k in tracked_layers},
    }
    prev_weights = {k: v.weight.data.clone() for k, v in tracked_layers.items()}

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 2), gamma=0.5)

    best_acc = -1.0
    best_model = copy.deepcopy(model.state_dict())
    for _ in range(epochs):
        tr_loss, tr_acc = _train_one_epoch(model, train_loader, criterion, optimizer, tracked_layers, history, device)
        te_loss, te_acc, _, _ = _evaluate(model, test_loader, criterion, device)
        _log_weight_stats(history, tracked_layers, prev_weights)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_loss"].append(te_loss)
        history["test_acc"].append(te_acc)
        if te_acc > best_acc:
            best_acc = te_acc
            best_model = copy.deepcopy(model.state_dict())
        scheduler.step()

    model.load_state_dict(best_model)
    torch.save(model.state_dict(), os.path.join(output_dir, "models", f"{config_name}_best.pth"))
    torch.save(model.state_dict(), os.path.join(output_dir, "models", f"{config_name}_final.pth"))
    export_to_onnx(model, os.path.join(output_dir, "models", f"{config_name}.onnx"), device)

    test_loss, test_acc, y_true, y_pred = _evaluate(model, test_loader, criterion, device)
    history["best_test_acc"] = float(best_acc)
    history["final_test_acc"] = float(test_acc)
    history["final_test_loss"] = float(test_loss)

    with open(os.path.join(output_dir, "training_history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    plot_training_curves(history, output_dir)
    plot_gradient_curves(history, output_dir)
    plot_weight_curves(history, output_dir)
    plot_gradient_heatmap(history, output_dir)
    plot_gradient_ratio(history, output_dir)
    plot_confusion(y_true, y_pred, output_dir)
    plot_sample_predictions(model, test_loader, output_dir, device)
    plot_sample_images(train_loader, output_dir)
    generate_summary_tables(history, output_dir, model_title)
    draw_architecture_svg(config_name, output_dir, model_title)
    return model, history

def summarize_last_epoch(history):
    rows = []
    for layer in history["tracked_layers"]:
        rows.append({
            "layer": layer,
            "grad_norm_last": history["epoch_grad_norms"][layer][-1],
            "weight_norm_last": history["weight_norms"][layer][-1],
            "weight_delta_last": history["weight_deltas"][layer][-1],
        })
    return rows

device = get_device()
print('Device:', device)

In [ ]:
# Model and tracking inspection
model = get_model('simple_cnn').to(device)
tracked = get_tracked_conv_layers(model)
print(model)
print('Tracked convolution layers:', len(tracked))
for i, name in enumerate(tracked.keys()):
    if i < 15:
        print('-', name)

In [ ]:
# Train and generate all artifacts
model, history = train_and_analyze(
    config_name='simple_cnn',
    output_dir='outputs/01_simple_cnn',
    model_title='Simple CNN (No Residual, No Skip)',
    epochs=20,
    batch_size=128,
    learning_rate=1e-3,
    data_dir='../dataset/SVHN',
)
print('Best test accuracy:', round(history['best_test_acc'], 4))
print('Final test accuracy:', round(history['final_test_acc'], 4))

In [ ]:
# Per-layer final epoch summary
rows = summarize_last_epoch(history)
print('Last-epoch layer summary (first 20 conv layers):')
for row in rows[:20]:
    print(row)
print('Total tracked conv layers:', len(rows))

In [ ]:
# Visual proof outputs
plot_dir = 'outputs/01_simple_cnn/plots'
to_show = [
    'accuracy_curves.png', 'loss_curves.png', 'epoch_gradient_norms.png',
    'batch_gradient_norms.png', 'gradient_heatmap.png', 'gradient_ratio.png',
    'weight_norms.png', 'weight_deltas.png', 'confusion_matrix.png',
    'sample_images.png', 'sample_predictions.png',
    'table1_architecture.png', 'table2_performance.png',
    'table3_training_metrics.png', 'table4_gradient_summary.png',
]
for name in to_show:
    p = os.path.join(plot_dir, name)
    if os.path.exists(p):
        print('Showing:', name)
        display(Image(filename=p))

print('Model artifacts:')
print('- outputs/01_simple_cnn/models/simple_cnn_best.pth')
print('- outputs/01_simple_cnn/models/simple_cnn_final.pth')
print('- outputs/01_simple_cnn/models/simple_cnn.onnx')
print('- outputs/01_simple_cnn/models/_simple_cnn.onnx.svg')